In [16]:
import numpy as np
from scipy.stats import chi2, norm

np.random.seed(42)
n = 100
alpha = 0.05

def merge_small_frequencies(obs, exp, edges):
    """Объединяет соседние интервалы, если эмпирическая частота < 5."""
    i = 0
    while i < len(obs) - 1:
        if obs[i] < 5:
            # Складываем с правым соседом
            obs[i] += obs[i+1]
            exp[i] += exp[i+1]
            # Удаляем правый сосед
            obs = np.delete(obs, i+1)
            exp = np.delete(exp, i+1)
            edges = np.delete(edges, i+1)
            # Не увеличиваем i, проверяем объединённый интервал снова
        else:
            i += 1
    # Проверка последнего интервала
    if len(obs) > 1 and obs[-1] < 5:
        obs[-2] += obs[-1]
        exp[-2] += exp[-1]
        obs = np.delete(obs, -1)
        exp = np.delete(exp, -1)
        edges = np.delete(edges, -1)
    return obs, exp, edges

print("\nЭкспоненциальное распределение (метод обратной функции)")

lambda_true = 0.1  # Параметр распределения
r = np.random.uniform(0, 1, n)
X_exp = - (1 / lambda_true) * np.log(r)

# 1. Статистические оценки и погрешности
mean_theo = 1 / lambda_true 
var_theo = 1 / lambda_true**2

mean_sample = np.mean(X_exp) # Выборочное среднее
var_sample = np.var(X_exp, ddof=1)  # Несмещённая оценка дисперсии

print(f"Теоретические значения: M(X) = {mean_theo:.4f}, D(X) = {var_theo:.4f}")
print(f"Выборочные оценки:      M(X) = {mean_sample:.4f}, D(X) = {var_sample:.4f}")
print(f"Абс. погрешности:       |M| = {abs(mean_sample - mean_theo):.4f}, |D| = {abs(var_sample - var_theo):.4f}")

# 2. Критерий Пирсона
k = int(round(1 + 3.3221 * np.log10(n))) 
h = (X_exp.max() - X_exp.min()) / k
edges = np.linspace(X_exp.min(), X_exp.max(), k + 1)
obs_freq, _ = np.histogram(X_exp, bins=edges) # Наблюдаемые частоты

lambda_hat = 1 / mean_sample
P_theo = np.exp(-lambda_hat * edges[:-1]) - np.exp(-lambda_hat * edges[1:])
exp_freq = n * P_theo # Теоретические частоты

print(f"\nДо объединения. \nНаблюдаемые частоты: {obs_freq} \nТеоретические частоты: {exp_freq}")

# Объединение малочисленных частот
obs_m, exp_m, edges_m = merge_small_frequencies(obs_freq.copy(), exp_freq.copy(), edges.copy())
k_merged = len(obs_m)

print(f"\nПосле объединения. \nНаблюдаемые частоты: {obs_m} \nТеоретические частоты: {exp_m}")

chi2_obs = np.sum((obs_m - exp_m)**2 / exp_m)
df = k_merged - 2  # Число степеней свободы для экспоненциального распределения
chi2_crit = chi2.ppf(1 - alpha, df)

print(f"\nКритерий Пирсона:")
print(f"Число интервалов: исходное = {k}, после объединения = {k_merged}")
print(f"hi^2_набл = {chi2_obs:.4f}")
print(f"hi^2_кр(α={alpha}, df={df}) = {chi2_crit:.4f}")
print(f"Вывод: {'Гипотеза о показательном распределении НЕ ОТВЕРГАЕТСЯ' if chi2_obs < chi2_crit else 'Гипотеза ОТВЕРГАЕТСЯ'}")




Экспоненциальное распределение (метод обратной функции)
Теоретические значения: M(X) = 10.0000, D(X) = 100.0000
Выборочные оценки:      M(X) = 10.9660, D(X) = 103.0354
Абс. погрешности:       |M| = 0.9660, |D| = 3.0354

До объединения. 
Наблюдаемые частоты: [46 24 11 10  5  3  0  1] 
Теоретические частоты: [44.09540501 24.41589512 13.51923028  7.48568039  4.14486696  2.29503816
  1.27077665  0.70363679]

После объединения. 
Наблюдаемые частоты: [46 24 11 10  9] 
Теоретические частоты: [44.09540501 24.41589512 13.51923028  7.48568039  8.41431856]

Критерий Пирсона:
Число интервалов: исходное = 8, после объединения = 5
hi^2_набл = 1.4441
hi^2_кр(α=0.05, df=3) = 7.8147
Вывод: Гипотеза о показательном распределении НЕ ОТВЕРГАЕТСЯ


In [17]:

print("\nНормальное распределение")
mu_true, sigma_true = 3.0, 0.25

# Метод 1: Центральная предельная теорема (сумма 12 равномерных СВ)
r = np.random.uniform(0, 1, (n, 12)) # Матрица 100 x 12
z = np.sum(r, axis=1) - 6
X = mu_true + sigma_true * z

mean_clt = np.mean(X) # Выборочное среднее
var_clt = np.var(X, ddof=1) # Выборочная несмещенная дисперсия
std_clt = np.sqrt(var_clt)

print(f"\nВыборочное среднее = {mean_clt}")
print(f"Выборочная дисперсия = {var_clt}")
print(f"Выборочное среднее кв.откл. = {std_clt}")

abs_error_mean_clt = abs(mean_clt - mu_true)
abs_error_var_clt = abs(var_clt - sigma_true**2)
abs_error_std_clt = abs(std_clt - sigma_true)

rel_error_mean = (abs_error_mean_clt / mu_true) * 100
rel_error_std = (abs_error_std_clt / sigma_true) * 100

print(f"\nПогрешность мат. ожидания = {abs_error_mean_clt}")
print(f"Погрешность дисперсии = {abs_error_var_clt}")
print(f"Погрешность ср. кв. отклонения: = {abs_error_std_clt}")

print(f"\nОтносительная погрешность мат. ожидания = {rel_error_mean}%")
print(f"Относительная погрешность ср. кв. отклонения = {rel_error_std}%")


Нормальное распределение

Выборочное среднее = 2.995499811037961
Выборочная дисперсия = 0.06806563222622114
Выборочное среднее кв.откл. = 0.2608939099063471

Погрешность мат. ожидания = 0.004500188962039076
Погрешность дисперсии = 0.00556563222622114
Погрешность ср. кв. отклонения: = 0.010893909906347088

Относительная погрешность мат. ожидания = 0.15000629873463583%
Относительная погрешность ср. кв. отклонения = 4.357563962538835%


In [18]:
# Метод 2: Метод Мюллера
r1, r2 = np.random.uniform(0, 1, n), np.random.uniform(0, 1, n)
z = np.sqrt(-2 * np.log(r1)) * np.cos(2 * np.pi * r2)
X_muller = mu_true + sigma_true * z

mean_muller = np.mean(X_muller)
var_muller = np.var(X_muller, ddof=1)
std_muller = np.sqrt(var_muller)

print("Метод Мюллера:")
print(f"Теоретические значения: M(X) = {mu_true:.4f}, D(X) = {sigma_true**2:.4f}, sigma = {sigma_true:.4f}")
print(f"Выборочное среднее = {mean_muller:.6f}")
print(f"Выборочная дисперсия = {var_muller:.6f}")
print(f"Выборочное ср. кв. откл. = {std_muller:.6f}")
print(f"Погрешность мат. ожидания = {abs(mean_muller - mu_true):.6f}")
print(f"Погрешность дисперсии = {abs(var_muller - sigma_true**2):.6f}")
print(f"Погрешность ср. кв. откл. = {abs(std_muller - sigma_true):.6f}")


Метод Мюллера:
Теоретические значения: M(X) = 3.0000, D(X) = 0.0625, sigma = 0.2500
Выборочное среднее = 3.007152
Выборочная дисперсия = 0.053521
Выборочное ср. кв. откл. = 0.231345
Погрешность мат. ожидания = 0.007152
Погрешность дисперсии = 0.008979
Погрешность ср. кв. откл. = 0.018655


In [19]:
# Критерий Пирсона — нормальное распределение
def pearson_test_normal(X_data, alpha_val, method_name):
    k_int = int(round(1 + 3.3221 * np.log10(n)))
    edges = np.linspace(X_data.min(), X_data.max(), k_int + 1)
    obs_freq, _ = np.histogram(X_data, bins=edges)
    midpoints = (edges[:-1] + edges[1:]) / 2

    # Выборочное среднее через середины интервалов
    x_bar = np.sum(midpoints * obs_freq) / n
    s = np.std(X_data, ddof=1)

    # Нормированные границы интервалов
    z = (edges - x_bar) / s
    z_left = z[:-1].copy(); z_left[0] = -np.inf
    z_right = z[1:].copy(); z_right[-1] = np.inf

    # Вероятности попадания в интервалы
    P = norm.cdf(z_right) - norm.cdf(z_left)
    exp_freq = n * P

    print(f"{method_name} — до объединения:")
    print(f"Наблюдаемые частоты:  {obs_freq}")
    print(f"Теоретические частоты: {np.round(exp_freq, 4)}")

    # Объединение малочисленных частот
    obs_m = obs_freq.copy().astype(float)
    exp_m = exp_freq.copy()
    i = 0
    while i < len(obs_m) - 1:
        if obs_m[i] < 5:
            obs_m[i+1] += obs_m[i]; exp_m[i+1] += exp_m[i]
            obs_m = np.delete(obs_m, i); exp_m = np.delete(exp_m, i)
        else:
            i += 1
    if len(obs_m) > 1 and obs_m[-1] < 5:
        obs_m[-2] += obs_m[-1]; exp_m[-2] += exp_m[-1]
        obs_m = obs_m[:-1]; exp_m = exp_m[:-1]
    k_merged = len(obs_m)

    print(f"{method_name} — после объединения:")
    print(f"Наблюдаемые частоты:  {obs_m.astype(int)}")
    print(f"Теоретические частоты: {np.round(exp_m, 4)}")

    chi2_obs = np.sum((obs_m - exp_m)**2 / exp_m)
    df = k_merged - 3
    chi2_crit_val = chi2.ppf(1 - alpha_val, df)

    print(f"Число интервалов: исходное = {k_int}, после объединения = {k_merged}")
    print(f"chi^2_набл = {chi2_obs:.4f}")
    print(f"chi^2_кр(alpha={alpha_val}, df={df}) = {chi2_crit_val:.4f}")
    if chi2_obs < chi2_crit_val:
        print("Вывод: Гипотеза о нормальном распределении НЕ ОТВЕРГАЕТСЯ")
    else:
        print("Вывод: Гипотеза о нормальном распределении ОТВЕРГАЕТСЯ")

pearson_test_normal(X, alpha, "ЦПТ (Метод 1)")
pearson_test_normal(X_muller, alpha, "Метод Мюллера (Метод 2)")


ЦПТ (Метод 1) — до объединения:
Наблюдаемые частоты:  [ 4 14 13 26 22 10  7  4]
Теоретические частоты: [ 5.1598  9.7773 17.7447 22.9524 21.161  13.9053  6.5118  2.7877]
ЦПТ (Метод 1) — после объединения:
Наблюдаемые частоты:  [18 13 26 22 10 11]
Теоретические частоты: [14.9371 17.7447 22.9524 21.161  13.9053  9.2995]
Число интервалов: исходное = 8, после объединения = 6
chi^2_набл = 3.7424
chi^2_кр(alpha=0.05, df=3) = 7.8147
Вывод: Гипотеза о нормальном распределении НЕ ОТВЕРГАЕТСЯ
Метод Мюллера (Метод 2) — до объединения:
Наблюдаемые частоты:  [ 5  8 21 17 24 16  5  4]
Теоретические частоты: [ 4.1398  8.8679 17.2374 23.3922 22.1652 14.6645  6.773   2.76  ]
Метод Мюллера (Метод 2) — после объединения:
Наблюдаемые частоты:  [ 5  8 21 17 24 16  9]
Теоретические частоты: [ 4.1398  8.8679 17.2374 23.3922 22.1652 14.6645  9.533 ]
Число интервалов: исходное = 8, после объединения = 7
chi^2_набл = 3.1350
chi^2_кр(alpha=0.05, df=4) = 9.4877
Вывод: Гипотеза о нормальном распределении НЕ ОТВЕРГА

In [20]:
# Критерий Колмогорова-Смирнова — нормальное распределение

def ks_test_normal(X_data, alpha_val, method_name):
    k_int = int(round(1 + 3.3221 * np.log10(n)))
    edges = np.linspace(X_data.min(), X_data.max(), k_int + 1)
    obs_freq, _ = np.histogram(X_data, bins=edges)
    midpoints = (edges[:-1] + edges[1:]) / 2

    x_bar = np.mean(X_data)
    s = np.std(X_data, ddof=1)

    cum_freq = np.cumsum(obs_freq)
    Fn = cum_freq / n

    # Теоретическая функция Лапласа: F(x) = 0.5 + Ф((x-m)/sigma)
    z_mid = (midpoints - x_bar) / s
    F_theo = norm.cdf(z_mid)  # norm.cdf = 0.5 + Ф(z)

    diff = np.abs(Fn - F_theo)
    D_max = np.max(diff)
    lambda_val = D_max * np.sqrt(n)
    lambda_cr = 1.36  # при alpha=0.05

    print(f"{method_name}:")
    print(f"{'xi':>10} {'ni':>5} {'n_нак':>7} {'Fn(x)':>8} {'F(x)':>8} {'|Fn-F|':>8}")
    for i in range(len(midpoints)):
        print(f"{midpoints[i]:>10.4f} {int(obs_freq[i]):>5} {int(cum_freq[i]):>7} {Fn[i]:>8.4f} {F_theo[i]:>8.4f} {diff[i]:>8.4f}")

    print(f"D_макс = {D_max:.4f}")
    print(f"lambda = D_макс * sqrt(n) = {lambda_val:.4f}")
    print(f"lambda_кр = {lambda_cr} (alpha={alpha_val})")
    if lambda_val > lambda_cr:
        print("Вывод: Гипотеза о нормальном распределении ОТВЕРГАЕТСЯ")
    else:
        print("Вывод: Гипотеза о нормальном распределении НЕ ОТВЕРГАЕТСЯ")

ks_test_normal(X, alpha, "ЦПТ (Метод 1)")
ks_test_normal(X_muller, alpha, "Метод Мюллера (Метод 2)")


ЦПТ (Метод 1):
        xi    ni   n_нак    Fn(x)     F(x)   |Fn-F|
    2.4982     4       4   0.0400   0.0283   0.0117
    2.6522    14      18   0.1800   0.0941   0.0859
    2.8062    13      31   0.3100   0.2341   0.0759
    2.9603    26      57   0.5700   0.4463   0.1237
    3.1143    22      79   0.7900   0.6756   0.1144
    3.2683    10      89   0.8900   0.8522   0.0378
    3.4224     7      96   0.9600   0.9491   0.0109
    3.5764     4     100   1.0000   0.9870   0.0130
D_макс = 0.1237
lambda = D_макс * sqrt(n) = 1.2371
lambda_кр = 1.36 (alpha=0.05)
Вывод: Гипотеза о нормальном распределении НЕ ОТВЕРГАЕТСЯ
Метод Мюллера (Метод 2):
        xi    ni   n_нак    Fn(x)     F(x)   |Fn-F|
    2.5358     5       5   0.0500   0.0208   0.0292
    2.6766     8      13   0.1300   0.0765   0.0535
    2.8174    21      34   0.3400   0.2061   0.1339
    2.9583    17      51   0.5100   0.4163   0.0937
    3.0991    24      75   0.7500   0.6544   0.0956
    3.2399    16      91   0.9100   0.842